In [37]:
from pathlib import Path
import json
import math
import struct

In [38]:
TARGET_NAME="LeNet-300-100"
TARGET_DIR="../results/1.LeNet-300-100"
TARGET_DATE="20260402"
TARGET_TYPE="canonical"
MODEL_ID1=1
MODEL_ID2=3
TARGET_CONFIG=f"freq_{MODEL_ID1}_{MODEL_ID2}.json"

In [39]:
CONFIG_PATH = Path(f"{TARGET_DIR}/{TARGET_DATE}/{TARGET_TYPE}/{TARGET_CONFIG}")

with open(CONFIG_PATH) as f:
    cfg = json.load(f)

best = cfg["best_encoding"]
HUFFMAN_CODES = best["huffman_codes"]  # symbol(str) → bitstring
MERGED_VALUES = set(best["merged_values"])
FALLBACK_SYMBOL = best["representative_value"] # special rare-delta token
FALLBACK_CODE = HUFFMAN_CODES[str(FALLBACK_SYMBOL)]
BITSLENGTH = best["bitsLen_symNum"]
MIN_BITS_LENGTH = best["min_code_length"]
MAX_BITS_LENGTH = best["max_code_length"]

# Build reverse map  bitstring → int symbol  (for decoding)
REVERSE_CODES = {v: int(k) for k, v in HUFFMAN_CODES.items()}

print(f"[config]  total symbols in Huffman table : {len(HUFFMAN_CODES)}")
print(f"[config]  merged (rare) values           : {len(MERGED_VALUES)}")
print(f"[config]  fallback code for 1000         : {FALLBACK_CODE}")
avg_bits = cfg["best_avg_bits"]
print(
    f"[config]  best avg bits/symbol           : {avg_bits:.4f}  "
    f"(vs 8 raw bits → {avg_bits/8*100:.1f}% of original)"
)

[config]  total symbols in Huffman table : 262
[config]  merged (rare) values           : 250
[config]  fallback code for 1000         : 000100
[config]  best avg bits/symbol           : 7.0287  (vs 8 raw bits → 87.9% of original)


In [40]:
print(BITSLENGTH)

{'4': 1, '6': 33, '7': 26, '8': 18, '9': 17, '10': 84, '11': 53, '12': 30}


In [41]:
def symbol_section_size(
    bitslen_symnum: dict[str, int],
    huffman_codes:  dict[str, str],
) -> int:
    """Return the byte size of the encoded symbol section (always a multiple of 8)."""
    n_counts  = len(bitslen_symnum)
    n_symbols = len(huffman_codes)
    total_bits = n_counts * 8 + n_symbols * 9
    return math.ceil(total_bits / 64) * 8   # round up to full 8-byte frames

# ─────────────────────────────────────────────────────────────────────────────
# 9-bit sign-magnitude helpers
# ─────────────────────────────────────────────────────────────────────────────

def encode_sym9(value: int) -> str:
    """
    Encode a symbol value (-255..255) as a 9-bit sign-magnitude bitstring.
        bit[0]   : sign  (0 = positive / zero, 1 = negative)
        bits[1-8]: magnitude as unsigned 8-bit
    e.g.  -7  →  1_00000111  →  "100000111"
          7   →  0_00000111  →  "000000111"
          0   →  0_00000000  →  "000000000"
    """
    if not (-255 <= value <= 255):
        raise ValueError(f"Symbol value {value} out of range -255..255")
    sign = 1 if value < 0 else 0
    mag  = abs(value)
    return f"{sign}{mag:08b}"


def decode_sym9(bits9: str) -> int:
    """
    Decode a 9-bit sign-magnitude bitstring back to an integer.
    """
    sign = int(bits9[0])
    mag  = int(bits9[1:], 2)
    return -mag if sign else mag


def symbol_section_size_from_counts(n_lengths: int, n_symbols: int) -> int:
    """Byte size of the section given raw counts (no dicts needed)."""
    total_bits = n_lengths * 8 + n_symbols * 9
    return math.ceil(total_bits / 64) * 8

In [54]:
# ─────────────────────────────────────────────────────────────────────────────
# Unified section encoder  (histogram counts + symbol values, single bitstream)
# ─────────────────────────────────────────────────────────────────────────────


def encode_symbol_table(
    bitslen_symnum: dict[str, int],  # {"3": 1, "4": 2, "5": 14, ...}
    huffman_codes: dict[str, str],  # {"0": "000", "-1": "0010", ...}
) -> bytes:
    """
    Encode histogram counts and symbol values into a single contiguous bitstream,
    padded to a multiple of 8 bytes (frames).

    Bitstream layout:
        [ count_0 (8 bits) | count_1 (8 bits) | ... | count_N (8 bits) ]   ← Section A
        [ sym_0   (9 bits) | sym_1   (9 bits) | ... | sym_M   (9 bits) ]   ← Section B
        [ 0-padding to next 8-byte boundary                            ]

    Counts are emitted in ascending code-length order.
    Symbols are emitted sorted by (code_length, integer_value).
    """

    bits = []

    # ── Section A: histogram counts, 8 bits each ──────────────────────────────
    filled_codelen = {
        str(k): bitslen_symnum.get(str(k), 0)
        for k in range(min(map(int, bitslen_symnum)), max(map(int, bitslen_symnum)) + 1)
    }
    sorted_lengths = sorted(filled_codelen.keys(), key=int)  # ["3","4","5",...]
    for length_key in sorted_lengths:
        count = filled_codelen[length_key]
        bits.append(f"{count:08b}")

    # ── Section B: symbol values, 9 bits each (sign-magnitude) ───────────────
    sorted_symbols = sorted(
        huffman_codes.keys(), key=lambda k: (len(huffman_codes[k]), int(k))
    )
    for sym_key in sorted_symbols:
        bits.append(encode_sym9(int(sym_key)))

    # ── Pad to 8-byte (64-bit) boundary ──────────────────────────────────────
    bitstring = "".join(bits)
    pad = (64 - len(bitstring) % 64) % 64
    bitstring += "0" * pad

    # ── Pack into bytes ───────────────────────────────────────────────────────
    return bytes(int(bitstring[i : i + 8], 2) for i in range(0, len(bitstring), 8))

In [55]:
# ─────────────────────────────────────────────────────────────────────────────
# Unified section decoder
# ─────────────────────────────────────────────────────────────────────────────

def decode_symbol_table(
    data:        bytes,
    n_lengths:   int,     # number of distinct code lengths  = max_code - min_code + 1
    n_symbols:   int,     # total number of Huffman symbols  = num_huffman_symbols
    min_code_len:int,     # from header — to reconstruct the length keys
) -> tuple[dict[int, int], bytes]:
    """
    Decode the unified symbol section from the front of `data`.

    Returns:
        (symbol_code_len, remainder)

        symbol_code_len : dict  int_symbol_value → code_length
        remainder       : bytes immediately after the symbol section
                          (i.e. the start of the bitstream payload)
    """

    # ── How many bytes does this section occupy? ──────────────────────────────
    section_bytes = symbol_section_size_from_counts(n_lengths, n_symbols)

    if len(data) < section_bytes:
        raise ValueError(
            f"Symbol section too short: need {section_bytes} bytes, "
            f"got {len(data)}."
        )

    # ── Unpack section bytes → full bitstring (drop padding implicitly) ───────
    raw      = data[:section_bytes]
    bitstring = "".join(f"{b:08b}" for b in raw)

    pos = 0

    # ── Section A: read n_lengths counts (8 bits each) ───────────────────────
    counts: dict[int, int] = {}
    for i in range(n_lengths):
        counts[min_code_len + i] = int(bitstring[pos : pos + 8], 2)
        pos += 8

    # ── Section B: read n_symbols values (9 bits each) ───────────────────────
    symbols: list[int] = []
    for _ in range(n_symbols):
        symbols.append(decode_sym9(bitstring[pos : pos + 9]))
        pos += 9

    # ── Reconstruct symbol → code_length mapping ─────────────────────────────
    symbol_code_len: dict[int, int] = {}
    sym_idx = 0
    for code_len in sorted(counts.keys()):
        count = counts[code_len]
        for sym in symbols[sym_idx : sym_idx + count]:
            symbol_code_len[sym] = code_len
        sym_idx += count

    remainder = data[section_bytes:]
    return symbol_code_len, remainder


In [56]:
print(len(BITSLENGTH))
print(len(HUFFMAN_CODES))

8
262


In [57]:
# Sort by (value, numeric key)
sorted_items = sorted(BITSLENGTH.items(), key=lambda x: (x[1], int(x[0])))

# Print result
for k, v in sorted_items:
    print(k, v)

4 1
9 17
8 18
7 26
12 30
6 33
11 53
10 84


In [58]:
symbol_section_size(BITSLENGTH, HUFFMAN_CODES)

304

In [59]:
symbol_section = encode_symbol_table(BITSLENGTH, HUFFMAN_CODES)
print(f"Encoded symbol table byte size: {len(symbol_section)}")

Encoded symbol table byte size: 304


In [60]:
symbol_section

b"\x01\x00!\x1a\x12\x11T5\x1e\x00p\xa2Q\x18\x84>\x1d\r\x86B\xe1P\x98D\x1e\r\x05\x82@\xe0P\x10\x08\x08\x06\x04\x02\x81\x80\xe0\x80H(\x16\x0c\x06\x83\xa3\xd1\xd8\xe4n5\x19\x8cE\xe2\xd1X\xa4L\x1e\x10\x08\x84\x82a@\xa8X.\x18\x0c\x86\x83a\xc9<\x9aK$\x91\xc8\xa42\x08\xf8t<\x1f\x10\x08DB1 \x96_.\x96\xcb%r\xa9L\xa0L'\x14\nEB\xb1`\xb4]\xf9\xfc}\xfe\xdf_\xa7\xcf\xe5\xf1\xf8{\xfd\xde\xdfg\xaf\xd5\xe9\xf4y\xfc\xde_'\x8f\xc5\xe1\xf0w\xfb\xdd\xde\xe7o\xb5\xd9\xecM\xe6\xb3I\x9c\xcac0\x17\x8c\x06#!\x98\xd0j\xd6k\xb6\x1b-\xa6\xdbq\xba\xdeo\xb8\x1c.'\x1b\x91\xca\xe6s\xba\x1d.\xa7[\xb1\xda\xeew\xbc\x1e/'\x9b\xd1\xea\xf6{\xbe\x1f?\xff\xef\xf3\xf7\xfa\xeb\xf5\xba\xbdN\x9fK\xa3\xd0\xe7\xf3\xb9\xbc\xce_\x1ey;\x9dNg\x13a\xb0\xdcp9\x1d\x0e\xc7\x83\xd1\xf3\x11\x8e\xc8e2\xd9\x8c\xd6s=\xa0\xd1i4\xda\x8dW\xd3\xed\xf8\xff\xffr\xb9<\x8e7\x17\x89\xc3\xe0\xf07\xdb\xba\r\x02\x7f>\x9e\x8f\xc8\x04\x12\x11\x0c\x88\xc2a\xb1X\xcc\x97\xeb\xf8"

In [61]:
for i in range(0, len(symbol_section), 8):
    chunk = symbol_section[i:i+8]
    print(' '.join(f'{b:02X}' for b in chunk))

01 00 21 1A 12 11 54 35
1E 00 70 A2 51 18 84 3E
1D 0D 86 42 E1 50 98 44
1E 0D 05 82 40 E0 50 10
08 08 06 04 02 81 80 E0
80 48 28 16 0C 06 83 A3
D1 D8 E4 6E 35 19 8C 45
E2 D1 58 A4 4C 1E 10 08
84 82 61 40 A8 58 2E 18
0C 86 83 61 C9 3C 9A 4B
24 91 C8 A4 32 08 F8 74
3C 1F 10 08 44 42 31 20
96 5F 2E 96 CB 25 72 A9
4C A0 4C 27 14 0A 45 42
B1 60 B4 5D F9 FC 7D FE
DF 5F A7 CF E5 F1 F8 7B
FD DE DF 67 AF D5 E9 F4
79 FC DE 5F 27 8F C5 E1
F0 77 FB DD DE E7 6F B5
D9 EC 4D E6 B3 49 9C CA
63 30 17 8C 06 23 21 98
D0 6A D6 6B B6 1B 2D A6
DB 71 BA DE 6F B8 1C 2E
27 1B 91 CA E6 73 BA 1D
2E A7 5B B1 DA EE 77 BC
1E 2F 27 9B D1 EA F6 7B
BE 1F 3F FF EF F3 F7 FA
EB F5 BA BD 4E 9F 4B A3
D0 E7 F3 B9 BC CE 5F 1E
79 3B 9D 4E 67 13 61 B0
DC 70 39 1D 0E C7 83 D1
F3 11 8E C8 65 32 D9 8C
D6 73 3D A0 D1 69 34 DA
8D 57 D3 ED F8 FF FF 72
B9 3C 8E 37 17 89 C3 E0
F0 37 DB BA 0D 02 7F 3E
9E 8F C8 04 12 11 0C 88
C2 61 B1 58 CC 97 EB F8


In [62]:
with open(f"{TARGET_DIR}/package_symbol_table_{MODEL_ID1}_{MODEL_ID2}.bin", "wb") as f:
    f.write(symbol_section)